# Feature Importance — SHAP vs. Model-Native
Compares XGBoost's built-in feature importance against SHAP global importance.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.data_loader import load_unsw
from src.preprocessing import Preprocessor
from src.models.model_factory import create_model
from src.xai.shap_explainer import SHAPExplainer
from src.utils.config_loader import load_config, load_feature_sets, get_active_features, resolve_path

config = load_config("../configs/config.yaml")
feature_sets = load_feature_sets("../configs/feature_sets.yaml")
features = get_active_features(config, feature_sets)

df = load_unsw(resolve_path(config["paths"]["unsw_train"]), resolve_path(config["paths"]["unsw_test"]))
pre = Preprocessor(feature_list=features).fit(df)
X, y = pre.transform(df)

model = create_model(config["model"]["type"], config["model"]["params"])
model.fit(X, y)

In [ ]:
native_importance = pd.Series(model.get_feature_importance(), index=features).sort_values(ascending=False)
native_importance.head(15)

In [ ]:
explainer = SHAPExplainer(model, features, background_samples=config["xai"]["shap_background_samples"])
shap_importance = explainer.global_importance(X)
shap_importance.head(15)

In [ ]:
comparison = pd.DataFrame({"native_rank": native_importance.rank(ascending=False), "shap_rank": shap_importance.rank(ascending=False)})
comparison["rank_diff"] = (comparison["native_rank"] - comparison["shap_rank"]).abs()
comparison.sort_values("rank_diff", ascending=False).head(10)